# Fill in a temperature map: SciPy vs fastscipy

Resample a 64 × 64 temperature grid onto a 192 × 192 grid using shape-preserving PCHIP interpolation. Show the original grid and the actual interpolated maps.

Run **Run → Run All Cells**. Ordinary SciPy and fastscipy run in separate cells, then a final
cell compares timings and shows each implementation's actual inputs and outputs. Calibration
targets about five seconds of ordinary computation, capped at 128 batches. No data downloads.

## Set up

Use Python 3.12–3.15. Install dependencies in a fresh kernel, then restart if needed:

```python
%pip install --only-binary=:all: fastscipy==0.1.0a4 matplotlib
```

`import fastscipy` loads the package; **`fastscipy.patch()` enables acceleration**. These
standalone notebooks need no local helper files. Keep the default sizes to limit memory.

In [ ]:
import os

# Apply before numerical imports; restart the kernel after changing these limits.
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[name] = "1"
os.environ["FASTSCIPY_NUM_THREADS"] = "1"

In [ ]:
from hashlib import sha256
from math import ceil
from statistics import median
from time import perf_counter
import platform

import numpy as np
import scipy
from scipy import interpolate
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import fastscipy

AREA = "interpolate"
PATH_KEY = "RegularGridInterpolator pchip"
CALLS_PER_JOB = 1
TARGET_SCIPY_SECONDS = 5.0
MAX_JOBS = 128
SEED = 42

assert 0 < TARGET_SCIPY_SECONDS <= 10
assert 1 <= MAX_JOBS <= 128
status = fastscipy.patch(components=[AREA], threads=1)
assert status.get(AREA) == "on", status
print(f"{platform.system()} {platform.machine()} · Python {platform.python_version()}")
print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · fastscipy {fastscipy.__version__}")
print("1 fastscipy native thread; BLAS limited to 1 thread")

## Prepare the task

The deterministic factory produces the same inputs in both runs. Input creation is outside
the timer. Only the first batch's data and small per-batch hashes are retained for comparison.
`run_job` contains the SciPy operation being accelerated; both timed cells call it unchanged.

In [ ]:
UNIT = "maps"

COARSE_SIZE = 64
FINE_SIZE = 192
AXIS = np.linspace(0, 1, COARSE_SIZE)
FINE_AXIS = np.linspace(0, 1, FINE_SIZE)
X, Y = np.meshgrid(AXIS, AXIS)
FINE_X, FINE_Y = np.meshgrid(FINE_AXIS, FINE_AXIS)
QUERIES = np.stack((FINE_Y.ravel(), FINE_X.ravel()), axis=-1)


def make_job(index):
    phase = index * 0.025
    values = 16 + 6 * np.sin(4 * X + phase) * np.cos(3 * Y)
    values += 8 * np.exp(-((X - 0.6) ** 2 + (Y - 0.5) ** 2) / 0.04)
    model = interpolate.RegularGridInterpolator((AXIS, AXIS), values, method="pchip")
    return values, model


def input_arrays(job):
    return AXIS, job[0], QUERIES


def run_job(job):
    return job[1](QUERIES).reshape(FINE_SIZE, FINE_SIZE)


def validate(job, output):
    assert output.shape == (FINE_SIZE, FINE_SIZE) and output.dtype == np.float64
    assert np.isfinite(output).all()
    # Both grids include the same four corner locations.
    np.testing.assert_allclose(output[np.ix_([0, -1], [0, -1])], job[0][np.ix_([0, -1], [0, -1])], atol=1e-12)
    assert output.min() >= job[0].min() - 1e-10
    assert output.max() <= job[0].max() + 1e-10

## Warm up and size the batch

Verify acceleration and output agreement, then estimate the batch count from three warmed
SciPy calls. These calibration calls are excluded from the timing table. The five-second
target is approximate and depends on machine load and the batch cap.

In [ ]:
def fingerprint(arrays):
    digest = sha256()
    for array in arrays:
        digest.update(str((array.shape, array.dtype.str)).encode())
        digest.update(array.tobytes())
    return digest.digest()


preview_job = make_job(0)
with fastscipy.disabled():
    reference = run_job(preview_job)
    validate(preview_job, reference)
before = fastscipy.stats().get((PATH_KEY, "fast"), 0)
accelerated = run_job(preview_job)
validate(preview_job, accelerated)
assert fastscipy.stats().get((PATH_KEY, "fast"), 0) - before == CALLS_PER_JOB, "Expected fast path was not used"
assert reference.shape == accelerated.shape and reference.dtype == accelerated.dtype
assert reference.tobytes() == accelerated.tobytes(), "Warmup outputs differ"
del reference, accelerated

calibration = []
with fastscipy.disabled():
    for _ in range(3):
        start = perf_counter()
        output = run_job(preview_job)
        calibration.append(perf_counter() - start)
        del output
job_count = min(MAX_JOBS, max(1, ceil(TARGET_SCIPY_SECONDS / median(calibration))))
print(f"Batch: {job_count} {UNIT}; approximate SciPy target: {TARGET_SCIPY_SECONDS:.0f} s")
del preview_job

## 1. Run ordinary SciPy

Watch this cell process the batch with acceleration disabled. Timers include the operation
and output allocation, but exclude input generation, validation, hashing, and progress updates.

In [ ]:
scipy_inputs, scipy_outputs = [], []
scipy_seconds = 0.0
scipy_job_count = job_count
before = fastscipy.stats()
progress = display(Markdown(f"Ordinary SciPy: **0 / {job_count} batches**…"), display_id=True)
with fastscipy.disabled():
    for index in range(job_count):
        job = make_job(index)
        start = perf_counter()
        output = run_job(job)
        scipy_seconds += perf_counter() - start
        validate(job, output)
        scipy_inputs.append(fingerprint(input_arrays(job)))
        scipy_outputs.append(fingerprint((output,)))
        if index == 0:
            scipy_example_job, scipy_example_output = job, output.copy()
        del job, output
        if (index + 1) % max(1, job_count // 10) == 0:
            progress.update(Markdown(f"Ordinary SciPy: **{index + 1} / {job_count} batches**…"))
after = fastscipy.stats()
assert after.get((PATH_KEY, "fast"), 0) == before.get((PATH_KEY, "fast"), 0)
assert after.get((PATH_KEY, "scipy"), 0) - before.get((PATH_KEY, "scipy"), 0) == CALLS_PER_JOB * job_count
progress.update(Markdown(f"**Ordinary SciPy finished:** {scipy_seconds:.3f} s of computation."))

## 2. Run fastscipy

The same operation and reproducible inputs, now accelerated. Generation and validation still
take time, so cell wall time is longer than the computation time shown below.

In [ ]:
assert fastscipy.status().get(AREA) == "on", "Run setup first"
fast_inputs, fast_outputs = [], []
fast_seconds = 0.0
fast_job_count = job_count
before = fastscipy.stats()
for index in range(job_count):
    job = make_job(index)
    start = perf_counter()
    output = run_job(job)
    fast_seconds += perf_counter() - start
    validate(job, output)
    fast_inputs.append(fingerprint(input_arrays(job)))
    fast_outputs.append(fingerprint((output,)))
    if index == 0:
        fast_example_job, fast_example_output = job, output.copy()
    del job, output
after = fastscipy.stats()
assert after.get((PATH_KEY, "fast"), 0) - before.get((PATH_KEY, "fast"), 0) == CALLS_PER_JOB * job_count
assert after.get((PATH_KEY, "error"), 0) == before.get((PATH_KEY, "error"), 0)
display(Markdown(f"**fastscipy finished:** {fast_seconds:.3f} s of computation, using 1 native thread."))

## 3. Compare

Check all input and output hashes before displaying the first measured batch's actual data.
Both implementations use the same plot scales. Sequential timing can be affected by machine
load; this example is one workload, not an average across SciPy or a promised minimum speedup.

The plots show the complete first measured map and the same output crop in both rows. Model construction is excluded from timing; the timed operation evaluates a prebuilt interpolator.

In [ ]:
assert scipy_job_count == fast_job_count == job_count
assert len(scipy_outputs) == job_count
assert scipy_inputs == fast_inputs, "Inputs changed; run both timed cells again"
assert scipy_outputs == fast_outputs, "Outputs differ"
assert scipy_example_output.tobytes() == fast_example_output.tobytes()
speedup = scipy_seconds / fast_seconds
display(
    Markdown(
        f"### {job_count} {UNIT}\n\n"
        "| Implementation | Total computation time |\n"
        "|:--|--:|\n"
        f"| Ordinary SciPy | {scipy_seconds:.3f} s |\n"
        f"| fastscipy, 1 thread | {fast_seconds:.3f} s |\n\n"
        f"**{speedup:.1f}× faster** on this run.\n\n"
        "Actual inputs and outputs from the first measured batch are shown below."
    )
)
examples = (
    ("SciPy", scipy_example_job, scipy_example_output),
    ("fastscipy", fast_example_job, fast_example_output),
)

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
low, high = scipy_example_job[0].min(), scipy_example_job[0].max()
for row, (label, job, output) in enumerate(examples):
    views = ((job[0], "coarse input"), (output, "PCHIP output"), (output[65:125, 65:125], "output detail"))
    for col, (values, title) in enumerate(views):
        image = axes[row, col].imshow(
            values, cmap="magma", vmin=low, vmax=high, origin="lower", interpolation="nearest"
        )
        axes[row, col].set_title(f"{label} · {title}")
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
fig.colorbar(image, ax=axes.ravel().tolist(), label="Temperature (°C)", shrink=0.8)
plt.show()

## Try your own data

Replace `make_job` with your data source and update `validate` for its numerical contract.
Re-run warmup before measuring different inputs: shape, dtype, layout, and library versions
can change performance or select a SciPy fallback. Shorten `TARGET_SCIPY_SECONDS` for a quicker
run. The figures show the first measured batch; correctness checks cover every batch.

[More examples](README.md) · [fastscipy](https://github.com/michael-L-i/fastscipy)